[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Continuous Integration


## What you will be able to do

Write a GitHub Actions workflow that runs a project's tests on every push and pull request, read it
as the data GitHub reads, and run its steps the way a runner does: a fresh clone, a new environment,
one command at a time, stopping at the first that fails. Read a failed run's log from the bottom,
test on several systems and Pythons with a matrix, and keep a failing change from being merged.


## The idea

### The problem

The project from the **Project Layout** notebook installs, and its tests pass on your computer. That
proves less than it seems. Your environment holds whatever you installed over the months, including
packages the project never declared. Your folder holds files you never committed. And the tests ran
because you remembered to run them, which somebody in a hurry does not.

So a change can pass on your computer and fail on every other, and nobody finds out until somebody
else pulls it and the tests break in their hands. The question is how to run the tests on every
change, somewhere clean, without relying on anybody's memory.

### What continuous integration is

> In **continuous integration**, a project's checks, its tests first among them, run on a server
> every time a change is pushed, so that a change is tested before it joins the rest. On GitHub the
> service is **GitHub Actions**. A **workflow** is a YAML file in `.github/workflows` that says when
> to run, the **events** such as `push` and `pull_request`, and what to run: **jobs**, and a job
> runs on a **runner**, a fresh virtual machine, as a list of **steps** in order. A step runs a
> shell command, or an **action**, a reusable step such as `actions/checkout`, which clones the
> repository. A step fails when its command ends with an **exit code** other than 0. The job stops
> there, and GitHub marks the run as failed, beside the commit that started it.

### Why it works that way

- **A fresh machine every time.** A runner starts with nothing from the last run, so a file that was
  never committed, or a package that was never declared, fails there while everything passes on your
  computer.
- **The exit code is the verdict.** pytest ends with 1 when a test fails, `bash -e` stops at that
  command, and the runner reports the step as failed. Whatever hides an exit code hides the failure.
- **Steps run in order, and stop at the first failure.** The steps after it are skipped, so the last
  step that ran is the one to read.
- **On every push, not when somebody remembers.** The events in the workflow start it, so a change is
  tested whether or not its author ran the tests.
- **The workflow is a file in the repository.** It is committed and reviewed like the code it tests,
  and GitHub reads it from the commit being tested.
- **YAML is indentation.** GitHub reads the file as nested mappings and lists, so a space in the
  wrong place changes what the file says, or makes it unreadable.

### Where you will meet this

This library's own repository has three workflows. `check.yml` checks every link and house rule on
every push and pull request, `notebooks.yml` runs the notebooks that a push changed, and all of them
once a week, and `pages.yml` publishes the site. pytest, pip and NumPy run their tests with GitHub
Actions on every pull request, and GitHub's tutorial, Building and testing Python, sets up Python,
installs dependencies and runs pytest in a workflow with the same parts as this notebook's. GitHub
Actions is free on GitHub's standard runners for public repositories. The badge near the top of many
READMEs shows the result of the project's latest run.

### What this notebook covers

- A workflow file, read as data
- The repository a runner starts from
- The job, run the way a runner runs it
- Exit codes, which decide whether a step passes
- A failed run, read from the bottom
- A matrix of systems and Pythons
- Required checks, and a badge
- A change, from your folder to a passing run
- Five errors: a line of YAML one space too deep, a step that hides its exit code, a file never
  committed, a dependency never declared, and a path written for another folder

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import yaml

workflow = yaml.safe_load("""
name: Tests
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v7
      - uses: actions/setup-python@v7
        with:
          python-version: "3.13"
      - run: python -m pip install -e ".[test]"
      - run: python -m pytest
""")

for name, job in workflow["jobs"].items():
    print(f"job {name}, on a fresh {job['runs-on']} machine:")
    for number, step in enumerate(job["steps"], start=1):
        print(f"  {number}. {step.get('uses') or step.get('run')}")
```

```
job test, on a fresh ubuntu-latest machine:
  1. actions/checkout@v7
  2. actions/setup-python@v7
  3. python -m pip install -e ".[test]"
  4. python -m pytest
```

A workflow is data: YAML that GitHub reads into mappings and lists, here one job of four steps. For
every push and pull request, a new machine clones the repository, sets up Python, installs the
project and runs its tests. When the command of a step fails, the run fails, beside the commit that
caused it.


## Setup

Eight imports, and two functions.

- `subprocess` runs git, `venv`, pip, bash and pytest as programs of their own
- `sys` names the notebook's own Python, which makes the runner's environment and runs pytest here
- `os` sets the variables those programs read, and builds the `PATH` of a step
- `re` takes the time a test run took out of pytest's report
- `itertools` lists the combinations of a matrix
- `Path` names the project's folders, and writes and reads its files
- `shutil` removes the runner's folder before every job, and the scratch folder at the end
- `yaml`, from PyYAML, reads a workflow file

`run` runs a command in a folder, `scratch` unless it is told otherwise, and returns its exit code
and what it printed, with that folder's full path and the time a test run took taken out. Its
`environment` replaces the variables the command gets, which the runner's steps use. `git` runs git
in the project's folder. Setup points git away from this computer's own settings, which could sign
commits or name branches differently, and gives the notebook's commits a practice author, since git
records one on every commit.


In [1]:
import itertools
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import yaml

SCRATCH = Path("scratch")
PROJECT = SCRATCH / "stations"
RUNNER = SCRATCH / "runner"
for folder in [PROJECT / ".github" / "workflows", PROJECT / "src" / "stations", PROJECT / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun
os.environ["COLUMNS"] = "80"                  # and print reports 80 characters wide
os.environ["GIT_CONFIG_GLOBAL"] = os.devnull  # git reads neither this computer's settings for its user
os.environ["GIT_CONFIG_NOSYSTEM"] = "1"       # nor its settings for every user
for role in ["AUTHOR", "COMMITTER"]:          # and puts a practice name on the notebook's commits
    os.environ[f"GIT_{role}_NAME"] = "Weather Stations"
    os.environ[f"GIT_{role}_EMAIL"] = "stations@example.com"


def run(*command, folder=SCRATCH, environment=None):
    """Run a command in a folder, and return its exit code and what it printed, less this computer's paths."""
    finished = subprocess.run([str(part) for part in command], cwd=folder, capture_output=True, text=True,
                              env=environment)
    printed = (finished.stdout + finished.stderr).replace(f"{Path(folder).resolve()}/", "")
    return finished.returncode, re.sub(r" in \d+\.\d+s\b", "", printed).rstrip()


def git(*arguments):
    """Run git in the project's folder."""
    return run("git", *arguments, folder=PROJECT)


print("ready:", PROJECT)


ready: scratch/stations


## Worked examples

### A workflow file, read as data

A workflow lives in the repository, in `.github/workflows`, under any name that ends in `.yml` or
`.yaml`. YAML writes nested data with indentation: `key: value` lines make a mapping, and lines that
start with `- ` make a list. This workflow runs the project's tests on every push to `main` and on
every pull request:


In [2]:
%%writefile scratch/stations/.github/workflows/tests.yml
name: Tests

on:
  push:
    branches: [main]
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v7

      - uses: actions/setup-python@v7
        with:
          python-version: "3.13"

      - name: Install the project
        run: python -m pip install -e ".[test]"

      - name: Run the tests
        run: python -m pytest

      - name: Build a wheel
        run: python -m pip wheel --no-deps -w dist .


Writing scratch/stations/.github/workflows/tests.yml


In [3]:
workflow = yaml.safe_load((PROJECT / ".github" / "workflows" / "tests.yml").read_text())

print("keys:", list(workflow))
print("events:", workflow[True])
print("the job runs on:", workflow["jobs"]["test"]["runs-on"])
for number, step in enumerate(workflow["jobs"]["test"]["steps"], start=1):
    print(f"step {number}:", step)


keys: ['name', True, 'jobs']
events: {'push': {'branches': ['main']}, 'pull_request': None}
the job runs on: ubuntu-latest
step 1: {'uses': 'actions/checkout@v7'}
step 2: {'uses': 'actions/setup-python@v7', 'with': {'python-version': '3.13'}}
step 3: {'name': 'Install the project', 'run': 'python -m pip install -e ".[test]"'}
step 4: {'name': 'Run the tests', 'run': 'python -m pytest'}
step 5: {'name': 'Build a wheel', 'run': 'python -m pip wheel --no-deps -w dist .'}


The file is a mapping of three keys. `jobs` holds one job, `test`, and its `steps` are a list of five
mappings. A step has `uses`, an action, with the action's inputs under `with`, or `run`, a shell
command, and it can have a `name` to show in the log. `@v7` picks the action's major version, the one
its README shows.

The second key came back as `True`, and not as `on`. PyYAML follows version 1.1 of YAML, in which a
bare `on` means true. GitHub reads the key as the word `on`, so the workflow is right as written,
and a program that reads workflows with PyYAML looks the events up under `True`.

### The repository a runner starts from

A runner never sees your folder, only the repository. Here is the project from the **Project
Layout** notebook, trimmed to its module and its tests, with the `pyproject.toml` and the
`.gitignore` it had there:


In [4]:
%%writefile scratch/stations/src/stations/__init__.py
"""Mean temperatures from the weather stations' readings."""


Writing scratch/stations/src/stations/__init__.py


In [5]:
%%writefile scratch/stations/src/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/src/stations/readings.py


In [6]:
%%writefile scratch/stations/tests/test_readings.py
from stations.readings import mean, to_fahrenheit


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


def test_boiling_point_in_fahrenheit():
    assert to_fahrenheit(100) == 212


Writing scratch/stations/tests/test_readings.py


In [7]:
%%writefile scratch/stations/pyproject.toml
[build-system]
requires = ["setuptools>=77.0.3"]
build-backend = "setuptools.build_meta"

[project]
name = "stations"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = []

[project.optional-dependencies]
test = ["pytest==8.4.2"]

[tool.pytest.ini_options]
testpaths = ["tests"]


Writing scratch/stations/pyproject.toml


In [8]:
%%writefile scratch/stations/.gitignore
.venv/
build/
dist/
*.egg-info/
.pytest_cache/


Writing scratch/stations/.gitignore


git turns the folder into a repository and commits every file that `.gitignore` does not leave out.
`git ls-files` lists what the commit holds, and `git status --short` lists what it does not:


In [9]:
git("init", "-q", "-b", "main")
git("add", ".")
code, printed = git("commit", "-q", "-m", "The stations project, with its workflow")
print("commit exit code:", code)

print("in the commit:", git("ls-files")[1].splitlines())
print("not committed:", git("status", "--short")[1].splitlines())


commit exit code: 0
in the commit: ['.github/workflows/tests.yml', '.gitignore', 'pyproject.toml', 'src/stations/__init__.py', 'src/stations/readings.py', 'tests/test_readings.py']
not committed: []


Six files, and nothing left out. On GitHub, `git push` sends the commit to the repository, and the
push is the event that starts the workflow. What a runner tests is this commit, and not the folder it
came from.

### The job, run the way a runner runs it

On GitHub, a runner is a new virtual machine, and it runs a job's steps in order. `actions/checkout`
clones the repository, fetching only the commit that started the run. `actions/setup-python` installs
the Python that `with` names and puts it first on the `PATH`, so that `python` in a later step means
that Python. A `run` step runs its command with `bash -e`, in the folder the checkout made.

This notebook cannot start a virtual machine, so the three functions below do what those steps do,
on this computer, in `scratch/runner`. The runner's Python is a new environment made from the
notebook's own Python, whatever version the workflow names. It gets pip, as a runner's Python has,
so that the workflow's `python -m pip` commands run unchanged. `PIP_DISABLE_PIP_VERSION_CHECK` and
`PIP_NO_COLOR` are pip's `--disable-pip-version-check` and `--no-color`, given as variables:


In [10]:
def check_out():
    """What actions/checkout does: a fresh clone of the last commit, with nothing left from an earlier job."""
    shutil.rmtree(RUNNER, ignore_errors=True)
    return run("git", "clone", "-q", "--depth", "1", PROJECT.resolve().as_uri(), RUNNER.name)


def set_up_python():
    """What actions/setup-python does: a Python for the job, with pip. Here, a new environment in the clone."""
    run(sys.executable, "-m", "venv", "--without-pip", ".venv", folder=RUNNER)
    return run(sys.executable, "-m", "pip", "--python", ".venv", "--disable-pip-version-check", "--no-color",
               "install", "-q", "pip==25.3", folder=RUNNER)


def run_step(script):
    """What a run step does: bash -e runs the script in the clone, with the job's Python first on the PATH."""
    settings = {**os.environ, "PATH": f"{(RUNNER / '.venv' / 'bin').resolve()}{os.pathsep}{os.environ['PATH']}",
                "PIP_DISABLE_PIP_VERSION_CHECK": "1", "PIP_NO_COLOR": "1"}
    return run("bash", "-e", "-c", script, folder=RUNNER, environment=settings)


`run_job` reads the workflow from the last commit, as GitHub does, and runs the steps of one job in
order with those functions, printing the exit code of every step. When a step fails, it prints the
lines of its log worth reading first, pytest's `E`, `FAILED` and `ERROR` lines, then the last line
GitHub adds to a failed step, and it skips the steps after it:


In [11]:
def run_job(name="test"):
    """Run a job of the committed workflow as a runner does: its steps in order, until one fails."""
    workflow = yaml.safe_load(git("show", "HEAD:.github/workflows/tests.yml")[1])
    failed = 0
    for number, step in enumerate(workflow["jobs"][name]["steps"], start=1):
        title = step.get("name") or step.get("uses") or step["run"]
        if failed:
            print(f"{number}. {title}: skipped")
            continue
        if step.get("uses", "").startswith("actions/checkout"):
            code, log = check_out()
        elif step.get("uses", "").startswith("actions/setup-python"):
            code, log = set_up_python()
        else:
            code, log = run_step(step["run"])
        print(f"{number}. {title}: exit code {code}")
        if code != 0:
            for line in log.splitlines():
                if line.startswith(("E ", "FAILED", "ERROR")):
                    print("   ", line)
            print(f"    Error: Process completed with exit code {code}.")
            failed = code
    print("the run:", "failed" if failed else "passed")


run_job()


1. actions/checkout@v7: exit code 0
2. actions/setup-python@v7: exit code 0
3. Install the project: exit code 0
4. Run the tests: exit code 0
5. Build a wheel: exit code 0
the run: passed


Every step ended with 0, so the run passed, and on GitHub a green check appears beside the commit.
Nothing came from the folder you work in: the tests ran on what `git clone` fetched, installed into
an environment that held nothing but pip.

### Exit codes decide

A runner judges a step by one number, the exit code of its command. `bash -e` stops a script at the
first command that fails, so the step ends with that command's exit code:


In [12]:
for script in ["true", "false", "false\necho 'never printed'", "echo 'printed first'\nexit 3"]:
    code, printed = run("bash", "-e", "-c", script)
    print(f"{script!r:<32} exit code {code} | printed: {printed!r}")


'true'                           exit code 0 | printed: ''
'false'                          exit code 1 | printed: ''
"false\necho 'never printed'"    exit code 1 | printed: ''
"echo 'printed first'\nexit 3"   exit code 3 | printed: 'printed first'


`true` ends with 0 and `false` with 1. After `false`, `echo` never ran, and the script ended with
the 1 that `false` returned. A script can also end with an exit code of its own, as `exit 3` does.
pytest ends with 0 when every test passes, 1 when a test fails, 2 when it cannot collect a test
file, 4 when a path it was given does not exist, and 5 when it finds no tests, as the **Your First
Test** notebook showed, so a runner fails the tests' step in every case but the first.

### A failed run, read from the bottom

Here is a change that breaks the project, a slip in `to_fahrenheit`, committed as if it were fine:


In [13]:
readings = PROJECT / "src" / "stations" / "readings.py"
readings.write_text(readings.read_text().replace("celsius * 9 / 5 + 32", "celsius * 9 / 5 + 23"))
git("commit", "-q", "-am", "Tidy the conversion")

run_job()


1. actions/checkout@v7: exit code 0
2. actions/setup-python@v7: exit code 0
3. Install the project: exit code 0
4. Run the tests: exit code 1
    E       assert 203.0 == 212
    E        +  where 203.0 = to_fahrenheit(100)
    FAILED tests/test_readings.py::test_boiling_point_in_fahrenheit - assert 203....
    Error: Process completed with exit code 1.
5. Build a wheel: skipped
the run: failed


Read a failed run from the bottom up. The run failed at `Run the tests`, and the step's last line,
`Error: Process completed with exit code 1.`, says only that its command failed. Above it, pytest's
short summary names the test that failed, cut to the width of the report, and the `E` lines show what
the test compared: 203.0, where 212 was expected, from `to_fahrenheit(100)`. The wheel was never
built. On GitHub, the page of a run opens the failed step's log for you, and in a terminal,
`gh run view` with `--log-failed` prints the logs of the failed steps and nothing else.

The fix is a commit like any other, and the run that follows it passes:


In [14]:
readings.write_text(readings.read_text().replace("celsius * 9 / 5 + 23", "celsius * 9 / 5 + 32"))
git("commit", "-q", "-am", "Fix the conversion")

run_job()


1. actions/checkout@v7: exit code 0
2. actions/setup-python@v7: exit code 0
3. Install the project: exit code 0
4. Run the tests: exit code 0
5. Build a wheel: exit code 0
the run: passed


### A matrix of systems and Pythons

A job on one runner shows that the project works on one system, with one Python. `strategy.matrix`
runs the job once for every combination of the values it lists, and `${{ matrix.python-version }}`
puts the value for the job being run into a step. Here the project's workflow tests two systems and
three Pythons:


In [15]:
%%writefile scratch/stations/.github/workflows/tests.yml
name: Tests

on:
  push:
    branches: [main]
  pull_request:

jobs:
  test:
    runs-on: ${{ matrix.os }}
    strategy:
      matrix:
        os: [ubuntu-latest, macos-latest]
        python-version: ["3.12", "3.13", "3.14"]
    steps:
      - uses: actions/checkout@v7

      - uses: actions/setup-python@v7
        with:
          python-version: ${{ matrix.python-version }}

      - name: Install the project
        run: python -m pip install -e ".[test]"

      - name: Run the tests
        run: python -m pytest

      - name: Build a wheel
        run: python -m pip wheel --no-deps -w dist .


Overwriting scratch/stations/.github/workflows/tests.yml


In [16]:
git("commit", "-q", "-am", "Test on two systems and three Pythons")
workflow = yaml.safe_load((PROJECT / ".github" / "workflows" / "tests.yml").read_text())
matrix = workflow["jobs"]["test"]["strategy"]["matrix"]

for values in itertools.product(*matrix.values()):
    print("test", dict(zip(matrix, values)))


test {'os': 'ubuntu-latest', 'python-version': '3.12'}
test {'os': 'ubuntu-latest', 'python-version': '3.13'}
test {'os': 'ubuntu-latest', 'python-version': '3.14'}
test {'os': 'macos-latest', 'python-version': '3.12'}
test {'os': 'macos-latest', 'python-version': '3.13'}
test {'os': 'macos-latest', 'python-version': '3.14'}


Six jobs, on six runners at once, with the same steps, and a different `runs-on` and
`python-version`. `fail-fast` is on unless the workflow turns it off, so when one job fails, GitHub
cancels the jobs of the matrix that are still running or waiting, and `fail-fast: false` lets every
job finish. A matrix makes at most 256 jobs in one run.

The versions are in quotes for a reason:


In [17]:
print(yaml.safe_load("python-version: [3.9, 3.10, 3.11]"))
print(yaml.safe_load('python-version: ["3.9", "3.10", "3.11"]'))


{'python-version': [3.9, 3.1, 3.11]}
{'python-version': ['3.9', '3.10', '3.11']}


Unquoted, `3.10` is a number, and the number 3.10 is 3.1, so a job would ask setup-python for Python
3.1. Quote every version, as the workflow does.

### Required checks, and a badge

A failed run marks the commit, and does nothing more: a pull request whose run failed can still be
merged. A repository's settings can require status checks before merging, in a branch protection rule
or a ruleset for `main`. Once the `test` jobs are required, GitHub merges a pull request into `main`
only after they pass.

A badge shows the latest result of a workflow to everybody who opens the README. Its address is the
page of the workflow file with `/badge.svg` after it, and here are the badges of two of this
library's own workflows, as Markdown:


In [18]:
owner, repository = "johnfisher-ai", "Python-Visual-Guides"

for workflow_file in ["check.yml", "notebooks.yml"]:
    page = f"https://github.com/{owner}/{repository}/actions/workflows/{workflow_file}"
    print(f"[![{workflow_file}]({page}/badge.svg)]({page})")


[![check.yml](https://github.com/johnfisher-ai/Python-Visual-Guides/actions/workflows/check.yml/badge.svg)](https://github.com/johnfisher-ai/Python-Visual-Guides/actions/workflows/check.yml)
[![notebooks.yml](https://github.com/johnfisher-ai/Python-Visual-Guides/actions/workflows/notebooks.yml/badge.svg)](https://github.com/johnfisher-ai/Python-Visual-Guides/actions/workflows/notebooks.yml)


A badge shows the latest run on the repository's default branch, and `?branch=` or `?event=` at the
end of its address picks another branch or event. A badge only reports, and it is the required check
that keeps a failing change out of `main`.

### A change, from your folder to a passing run

The pieces of this notebook, in the order a change goes through them. `test_here` runs the tests the
way you run them where you work: the notebook's own Python and pytest, with `src` put on the import
path by `PYTHONPATH` and nothing installed, and `PYTEST_DISABLE_PLUGIN_AUTOLOAD` for the reason the
**Your First Test** notebook gave. A new function and its test are tried there first, then committed,
and the job runs on the commit, from the workflow with the matrix:


In [19]:
def test_here():
    """The tests as you run them where you work: this Python, its pytest, and the folder's own files."""
    settings = {**os.environ, "PYTHONPATH": "src", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}
    return run(sys.executable, "-m", "pytest", "-q", "--no-header", folder=PROJECT, environment=settings)


addition = ('\n\ndef to_kelvin(celsius):\n'
            '    """A temperature in degrees Celsius, in kelvin."""\n'
            '    return celsius + 273.15\n')
readings.write_text(readings.read_text() + addition)
tests = PROJECT / "tests" / "test_readings.py"
tests.write_text(tests.read_text().replace("import mean, to_fahrenheit", "import mean, to_fahrenheit, to_kelvin")
                 + "\n\ndef test_boiling_point_in_kelvin():\n    assert to_kelvin(100) == 373.15\n")

print("here:", test_here()[1].splitlines()[-1])
print("to commit:", git("status", "--short")[1].splitlines())
git("commit", "-q", "-am", "Convert to kelvin")
run_job()


here: 3 passed
to commit: [' M src/stations/readings.py', ' M tests/test_readings.py']
1. actions/checkout@v7: exit code 0
2. actions/setup-python@v7: exit code 0
3. Install the project: exit code 0
4. Run the tests: exit code 0
5. Build a wheel: exit code 0
the run: passed


The change passed where it was written, and then on a clean runner, from nothing but the commit. On
GitHub the same commit starts six jobs, one for every combination in the matrix, and a pull request
that holds it can be merged once the required checks pass.

### Where each part came from

| In the run | What it relies on | The section that showed it |
|---|---|---|
| `tests.yml`, read from the commit | a workflow, which is data in YAML | A workflow file, read as data |
| a clone with `to_kelvin` in it | a runner that sees only what was committed | The repository a runner starts from |
| checkout, Python, install, tests, wheel | steps run in order, until one fails | The job, run the way a runner runs it |
| `the run: passed` | every step ending with exit code 0 | Exit codes decide |
| the `E` and `FAILED` lines a failure would print | a log read from the bottom | A failed run, read from the bottom |
| six jobs on GitHub | two systems and three Pythons, quoted | A matrix of systems and Pythons |

Tests that pass here say the change works in your folder. The run says it works in the commit, which
is what everybody else gets.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/10-continuous-integration-solutions.ipynb).

**1.** Load the project's `.github/workflows/tests.yml` with `yaml.safe_load`, and print the name and
the command of every step that has a `run`.


In [20]:
# your code here


**2.** Run the script `"false\necho done"` with `bash -e -c`, then with `bash -c`, and print the exit
code and the output of both.


In [21]:
# your code here


**3.** Make a git repository in `scratch/task-repo`, commit one file, write a second file without
committing it, clone the repository into `scratch/task-clone`, and print the files in both folders.


In [22]:
# your code here


**4.** Write a test file with one failing test into `scratch/task-tests`, run pytest there with
`bash -e -c`, once plain and once followed by `|| true`, and print both exit codes.


In [23]:
# your code here


**5.** Add `windows-latest` to the systems in the matrix of the project's workflow, and print how
many jobs the matrix would make, and which.


In [24]:
# your code here


**6.** Print the Markdown for a badge of `notebooks.yml` in `johnfisher-ai/Python-Visual-Guides` that
shows only the runs on `main`.


In [25]:
# your code here


## Common errors

### yaml.scanner.ScannerError: mapping values are not allowed here


In [26]:
broken = (PROJECT / ".github" / "workflows" / "tests.yml").read_text().replace(
    "        run: python -m pytest", "         run: python -m pytest")

try:
    yaml.safe_load(broken)
except yaml.YAMLError as error:
    print(f"{type(error).__module__}.{type(error).__name__}: {error}")


yaml.scanner.ScannerError: mapping values are not allowed here
  in "<unicode string>", line 26, column 13:
             run: python -m pytest
                ^


`run:` sits one space deeper than `name:` above it, so YAML reads the line as more of the value `Run
the tests`, and a plain value cannot hold a colon followed by a space. The message names the line and
the column. GitHub reads the file the same way and cannot run it, and while an invalid workflow sits
in `.github/workflows`, every new commit gets a failed run. The keys of one step start in the same
column. Load a workflow with `yaml.safe_load` before you commit it:


In [27]:
fixed = yaml.safe_load((PROJECT / ".github" / "workflows" / "tests.yml").read_text())

print([step["name"] for step in fixed["jobs"]["test"]["steps"] if "run" in step])


['Install the project', 'Run the tests', 'Build a wheel']


### No error, and a passing run for failing tests: a step that ends with `|| true`


In [28]:
workflow_file = PROJECT / ".github" / "workflows" / "tests.yml"
workflow_file.write_text(workflow_file.read_text().replace("run: python -m pytest", "run: python -m pytest || true"))
readings.write_text(readings.read_text().replace("celsius * 9 / 5 + 32", "celsius * 9 / 5 + 23"))
git("commit", "-q", "-am", "Tidy the conversion, and keep the tests from blocking the build")

run_job()
code, log = run_step("python -m pytest || true")
print("the tests' step, again: exit code", code, "|", log.splitlines()[-1].strip("= "))


1. actions/checkout@v7: exit code 0
2. actions/setup-python@v7: exit code 0
3. Install the project: exit code 0
4. Run the tests: exit code 0
5. Build a wheel: exit code 0
the run: passed
the tests' step, again: exit code 0 | 1 failed, 2 passed


The run passed, and the wheel was built, from code whose tests fail. `||` runs `true` when
`python -m pytest` fails, and `true` always ends with 0, so the step ends with 0 whatever the tests
found. The failure is still in the log, where nobody looks at a run that passed. `|| true` usually
arrives as a way to get a build through while a test is broken, and stays. Take it out, and the run
fails as it should, until the conversion is fixed:


In [29]:
workflow_file.write_text(workflow_file.read_text().replace("run: python -m pytest || true", "run: python -m pytest"))
git("commit", "-q", "-am", "Let the tests fail the run")
run_job()

readings.write_text(readings.read_text().replace("celsius * 9 / 5 + 23", "celsius * 9 / 5 + 32"))
git("commit", "-q", "-am", "Fix the conversion")
print("here:", test_here()[1].splitlines()[-1])


1. actions/checkout@v7: exit code 0
2. actions/setup-python@v7: exit code 0
3. Install the project: exit code 0
4. Run the tests: exit code 1
    E       assert 203.0 == 212
    E        +  where 203.0 = to_fahrenheit(100)
    FAILED tests/test_readings.py::test_boiling_point_in_fahrenheit - assert 203....
    Error: Process completed with exit code 1.
5. Build a wheel: skipped
the run: failed
here: 3 passed


### ModuleNotFoundError: No module named 'stations.checks'


In [30]:
%%writefile scratch/stations/src/stations/checks.py
"""Checks on a reading before it is used."""


def is_plausible(celsius):
    """Whether a reading in degrees Celsius could be a real air temperature."""
    return -90 <= celsius <= 60


Writing scratch/stations/src/stations/checks.py


In [31]:
%%writefile scratch/stations/tests/test_checks.py
from stations.checks import is_plausible


def test_an_arctic_reading_is_plausible():
    assert is_plausible(-40)


def test_a_boiling_reading_is_not():
    assert not is_plausible(100)


Writing scratch/stations/tests/test_checks.py


In [32]:
git("add", "tests/test_checks.py")
git("commit", "-q", "-m", "Check readings")

print("here:", test_here()[1].splitlines()[-1])
run_job()


here: 5 passed
1. actions/checkout@v7: exit code 0
2. actions/setup-python@v7: exit code 0
3. Install the project: exit code 0
4. Run the tests: exit code 2
    E   ModuleNotFoundError: No module named 'stations.checks'
    ERROR tests/test_checks.py
    Error: Process completed with exit code 2.
5. Build a wheel: skipped
the run: failed


The tests pass here, where `src/stations/checks.py` sits in the folder, and fail on the runner, which
cloned the commit. `git add tests/test_checks.py` added the test and not the module it imports, so
the commit holds a test of a file that is not in it. `git status` lists the file that stayed behind,
and committing it fixes the run:


In [33]:
print("not committed:", git("status", "--short")[1].splitlines())
git("add", "src/stations/checks.py")
git("commit", "-q", "-m", "Add the module the checks test")

run_job()


not committed: ['?? src/stations/checks.py']
1. actions/checkout@v7: exit code 0
2. actions/setup-python@v7: exit code 0
3. Install the project: exit code 0
4. Run the tests: exit code 0
5. Build a wheel: exit code 0
the run: passed


### ModuleNotFoundError: No module named 'yaml'


In [34]:
%%writefile scratch/stations/src/stations/config.py
"""The list of stations, read from YAML."""

import yaml


def station_names(text):
    """The names in a YAML list of stations, such as 'stations: [Bergen, Oslo]'."""
    return yaml.safe_load(text)["stations"]


Writing scratch/stations/src/stations/config.py


In [35]:
%%writefile scratch/stations/tests/test_config.py
from stations.config import station_names


def test_two_station_names():
    assert station_names("stations: [Bergen, Oslo]") == ["Bergen", "Oslo"]


Writing scratch/stations/tests/test_config.py


In [36]:
git("add", ".")
git("commit", "-q", "-m", "Read the list of stations")

print("here:", test_here()[1].splitlines()[-1])
run_job()


here: 6 passed
1. actions/checkout@v7: exit code 0
2. actions/setup-python@v7: exit code 0
3. Install the project: exit code 0
4. Run the tests: exit code 2
    E   ModuleNotFoundError: No module named 'yaml'
    ERROR tests/test_config.py
    Error: Process completed with exit code 2.
5. Build a wheel: skipped
the run: failed


PyYAML is installed in the Python you work with, so the tests pass here. The runner's environment
holds only what `pyproject.toml` asks for, and its `dependencies` list is empty, so `import yaml`
fails there. Nothing is wrong with the code: the record of what the code needs is incomplete. Declare
the dependency, as a range, as the **Project Layout** notebook said a project states what it needs:


In [37]:
pyproject = PROJECT / "pyproject.toml"
pyproject.write_text(pyproject.read_text().replace("dependencies = []", 'dependencies = ["PyYAML>=6.0"]'))
git("commit", "-q", "-am", "Declare PyYAML")

run_job()


1. actions/checkout@v7: exit code 0
2. actions/setup-python@v7: exit code 0
3. Install the project: exit code 0
4. Run the tests: exit code 0
5. Build a wheel: exit code 0
the run: passed


### ERROR: file or directory not found: stations/tests


In [38]:
workflow_file.write_text(workflow_file.read_text().replace("run: python -m pytest", "run: python -m pytest stations/tests"))
git("commit", "-q", "-am", "Name the folder of the tests")

run_job()


1. actions/checkout@v7: exit code 0
2. actions/setup-python@v7: exit code 0
3. Install the project: exit code 0
4. Run the tests: exit code 4
    ERROR: file or directory not found: stations/tests
    Error: Process completed with exit code 4.
5. Build a wheel: skipped
the run: failed


`python -m pytest stations/tests` is how the tests run from the folder above the project, where the
command was tried first. A runner starts every `run` step in the root of the clone, where the folder
is plain `tests`, and pytest ends with exit code 4 when a path it was given does not exist. Write
paths from the root of the repository, or give the step a `working-directory:`, the folder its
command runs in:


In [39]:
workflow_file.write_text(workflow_file.read_text().replace("run: python -m pytest stations/tests", "run: python -m pytest"))
git("commit", "-q", "-am", "Run the tests from the root")

run_job()


1. actions/checkout@v7: exit code 0
2. actions/setup-python@v7: exit code 0
3. Install the project: exit code 0
4. Run the tests: exit code 0
5. Build a wheel: exit code 0
the run: passed


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the
repository and the runner's clone:


In [40]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A workflow in `.github/workflows` says when to run, under `on`, and what to run: jobs made of
  steps, on a fresh runner for every push or pull request.
- A runner starts from a clone of the commit, so it tests only what was committed, in an environment
  with only what the project declares.
- A step fails when its command ends with an exit code other than 0. The job stops there, and the
  steps after it are skipped.
- Read a failed run from the bottom: the failed step, pytest's short summary, then the `E` lines.
- Run the tests where you work before you push, and let the run check what your computer cannot: the
  commit, and the dependencies the project declares.
- A matrix runs the job for every combination of its values. Quote versions, since YAML reads `3.10`
  as 3.1.
- A required status check keeps a pull request from being merged until its run passes, and a badge
  shows the latest result.
- Never hide an exit code: `|| true` turns every failure into a pass.


## What is next

That is the end of this guide. You can write a test that catches a bug before it comes back, run a
suite with pytest and read its report, structure tests so that a failure says what broke, share
setup with fixtures, run one test over many inputs, check that code fails the right way, give a
project an environment of its own, record that environment so somebody else can rebuild it, lay the
project out so that it installs, and run its tests on every push.

The **NumPy, Deep Dive** guide comes next. It leaves the tools around a project for a library, NumPy,
and the shift from looping over values to thinking in arrays, including the shape errors and how to
read them. Its notebooks, like this guide's, are run by a workflow in this library's repository
whenever a push changes them.


---

&#8592; **Previous:** [Project Layout](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/09-project-layout.ipynb)  &nbsp;·&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
